In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

In [0]:
catalog = 'fmcg'
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'

In [0]:
## Enter manually the S3 uri to access data.

dbutils.widgets.text("data_source", "s3 uri", "Data Source")
# e.g. s3://bucket_name/customers/*.csv
data_source = dbutils.widgets.get("data_source")
print(data_source)

## Bronze

In [0]:
df_bronze = spark.read.csv(data_source, header=True, inferSchema=True)
display(df_bronze.limit(5))

In [0]:
df_bronze = df_bronze.withColumn('ingested_at', F.current_timestamp())\
    .withColumn('_source_file', F.col('_metadata.file_path'))\
    .withColumn('_file_size', F.col('_metadata.file_size'))\
    .withColumn('product_id', F.col('product_id').cast(StringType()))

df_bronze.printSchema()

In [0]:
df_bronze.write.mode('overwrite')\
    .format('delta')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{bronze_schema}.dim_gross_price')

## Silver

In [0]:
df_bronze = spark.table(f'{catalog}.{bronze_schema}.dim_gross_price')
display(df_bronze.limit(10))

In [0]:
df_bronze.select('month').distinct().show()

In [0]:
df_silver = df_bronze.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)
df_silver.select('month').distinct().show()

In [0]:
df_silver.select("gross_price").show(10)

In [0]:
# We are validating the gross_price column, converting only valid numeric values to double, fixing negative prices by making them positive, and replacing all non-numeric values with 0


df_silver = df_silver.withColumn(
    'gross_price',
    F.when(F.col('gross_price').rlike(r'^-?\d+(\.\d+)?$'), 
           F.when(F.col('gross_price').cast(FloatType()) < 0, -1 * F.col('gross_price').cast(FloatType()))
            .otherwise(F.col('gross_price').cast(FloatType())))
    .otherwise(0)
)
df_silver.select('gross_price').show(10)

In [0]:
# We enrich the silver dataset by performing an inner join with the products table to fetch the correct product_code for each product_id.

df_products = spark.table(f'{catalog}.{gold_schema}.child_dim_products') 
df_joined = df_silver.join(df_products.select('product_id', 'product_code'), on='product_id', how='inner')
df_joined = df_joined.select('product_id', 'product_code', 'month', 'gross_price', 'ingested_at', '_source_file', '_file_size')

df_joined.show(5)

In [0]:
df_joined.write.mode('overwrite')\
    .format('delta')\
    .option('mergeSchema', 'true')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{silver_schema}.dim_gross_price')

## Gold

In [0]:
df_silver = spark.table(f'{catalog}.{silver_schema}.dim_gross_price')

In [0]:
df_gold = df_silver.withColumn('year', F.year('month'))\
    .withColumnRenamed('gross_price', 'price_inr')

In [0]:
df_gold = df_gold.select('product_code', 'year', 'price_inr')
df_gold.show(5)

In [0]:
df_gold.write.mode('overwrite')\
    .format('delta')\
    .option('mergeSchema', 'true')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{gold_schema}.child_dim_gross_price')

## Merging Child Company with Parent

In [0]:

%sql
WITH price_per_year AS(
  SELECT
    product_code,
    price_inr,
    year,
    ROW_NUMBER() OVER(PARTITION BY product_code ORDER BY year DESC) AS rnk
  FROM
    fmcg.gold.child_dim_gross_price
),

latest_price AS(
  SELECT product_code, price_inr, year FROM price_per_year WHERE rnk = 1
)

MERGE INTO fmcg.gold.dim_gross_price AS target
USING latest_price AS source
ON target.product_code = source.product_code
WHEN MATCHED THEN
  UPDATE SET
    price_inr = source.price_inr,
    year = source.year
WHEN NOT MATCHED THEN
  INSERT (product_code, price_inr, year)
  VALUES (source.product_code, source.price_inr, source.year)